<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-11-cost-and-performance/lesson-11.6-vllm-and-gke-production/notebooks/exercises-11_6.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 11.6: Production Self-Hosting with vLLM on GKE

*Module 11 · Cost Optimization & Efficiency*

> Ollama is great for your laptop. For production — 50+ concurrent users, auto-scaling, zero downtime deploys, GPU utilization monitoring — you need vLLM on Kubernetes. This lesson deploys a production-grade, auto-scaling LLM serving stack on GKE with NVIDIA GPUs.

---

## About this notebook

This notebook contains the **7 runnable code examples** from the published lesson `lesson-11.6.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — GKE Cluster + GPU Node Pool — `01_cluster_setup.sh`
2. Step 2 — vLLM Deployment — Serve Models with OpenAI-Compatible API — `02_vllm_deployment.yaml`
3. Step 2 — vLLM Deployment — Serve Models with OpenAI-Compatible API — `deploy_vllm.sh`
4. Step 3 — Auto-Scaling — Scale on GPU Utilization + Queue Depth — `03_autoscaling.yaml`
5. Step 3 — Auto-Scaling — Scale on GPU Utilization + Queue Depth — `scaling_setup.sh`
6. Step 4 — Load Balancing + Ingress — HTTPS with Health Checks — `04_ingress.yaml`
7. Step 5 — Python Client — Use OpenAI SDK to Talk to vLLM — `vllm_client.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q openai requests


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · GKE Cluster + GPU Node Pool

Create the cluster, attach NVIDIA L4 GPUs, install the GPU driver DaemonSet.

**`01_cluster_setup.sh`** · _Block 1 of 7_

!/bin/bash — ═══════════════════════════════════════════


In [ ]:
#!/bin/bash
# ═══════════════════════════════════════════
# GKE CLUSTER + GPU NODE POOL
# NVIDIA L4 GPUs with cluster autoscaler.
# ═══════════════════════════════════════════

PROJECT_ID="your-project-id"
REGION="asia-south1"
ZONE="asia-south1-a"
CLUSTER="ai-serving-cluster"

# ── Step 1: Create GKE cluster ──
gcloud container clusters create ${CLUSTER} \
  --project ${PROJECT_ID} \
  --zone ${ZONE} \
  --machine-type e2-standard-4 \
  --num-nodes 1 \
  --enable-autoscaling --min-nodes 1 --max-nodes 3 \
  --release-channel regular \
  --workload-pool ${PROJECT_ID}.svc.id.goog

# ── Step 2: Add GPU node pool ──
gcloud container node-pools create gpu-pool \
  --cluster ${CLUSTER} \
  --zone ${ZONE} \
  --machine-type g2-standard-8 \
  --accelerator type=nvidia-l4,count=1 \
  --num-nodes 1 \
  --enable-autoscaling \
  --min-nodes 0 \
  --max-nodes 4 \
  --spot  # ← 60-70% cheaper with spot instances!

# ── Step 3: Install NVIDIA GPU drivers ──
kubectl apply -f https://raw.githubusercontent.com/GoogleCloudPlatform/container-engine-accelerators/master/nvidia-driver-installer/cos/daemonset-preloaded-latest.yaml

# ── Step 4: Verify GPU is available ──
kubectl get nodes -l cloud.google.com/gke-accelerator=nvidia-l4
kubectl describe node <gpu-node-name> | grep nvidia.com/gpu

echo "
  Cluster: ${CLUSTER}
  GPU: NVIDIA L4 (24GB VRAM)
  Node pool: 0-4 nodes (auto-scales)
  Spot pricing: ~₹25/hour (~₹18,000/month for 1 GPU)
  On-demand:    ~₹60/hour (~₹43,000/month)
"


> **What just happened?** A GKE cluster with a GPU node pool: g2-standard-8 machines with NVIDIA L4 GPUs (24 GB VRAM — enough for 8B-14B models). The node pool auto-scales from 0 to 4 nodes — when no requests, it scales to zero (pay nothing). --spot flag uses preemptible instances at 60-70% discount : ~₹18,000/month instead of ₹43,000. The NVIDIA driver DaemonSet installs GPU drivers automatically on each node.


### Step 2 · vLLM Deployment — Serve Models with OpenAI-Compatible API

Deploy vLLM as a Kubernetes Deployment with GPU resource requests.

**`02_vllm_deployment.yaml`** · _Block 2 of 7_

═══════════════════════════════════════════ — vLLM DEPLOYMENT


In [ ]:
# ═══════════════════════════════════════════
# vLLM DEPLOYMENT
# Serves google/gemma-2-9b-it with
# OpenAI-compatible API on port 8000.
# ═══════════════════════════════════════════
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-server
  labels:
    app: vllm-server
spec:
  replicas: 1
  selector:
    matchLabels:
      app: vllm-server
  template:
    metadata:
      labels:
        app: vllm-server
    spec:
      # Schedule on GPU nodes only
      nodeSelector:
        cloud.google.com/gke-accelerator: nvidia-l4
      tolerations:
        - key: nvidia.com/gpu
          operator: Exists
          effect: NoSchedule

      containers:
        - name: vllm
          image: vllm/vllm-openai:latest
          args:
            - "--model"
            - "google/gemma-2-9b-it"
            - "--max-model-len"
            - "8192"
            - "--gpu-memory-utilization"
            - "0.90"
            - "--dtype"
            - "auto"
            - "--enforce-eager"  # disable CUDA graphs for L4 compat
            - "--port"
            - "8000"

          ports:
            - containerPort: 8000
              name: http
              protocol: TCP

          resources:
            limits:
              nvidia.com/gpu: "1"   # request 1 GPU
              memory: "24Gi"
              cpu: "6"
            requests:
              nvidia.com/gpu: "1"
              memory: "16Gi"
              cpu: "4"

          # Health checks
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 120  # model loading takes ~2 min
            periodSeconds: 10
            failureThreshold: 3

          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 180
            periodSeconds: 30
            failureThreshold: 5

          env:
            - name: HUGGING_FACE_HUB_TOKEN
              valueFrom:
                secretKeyRef:
                  name: hf-token
                  key: token

      # Graceful shutdown: finish in-flight requests
      terminationGracePeriodSeconds: 120

---
# ── Service: expose vLLM within the cluster ──
apiVersion: v1
kind: Service
metadata:
  name: vllm-service
spec:
  selector:
    app: vllm-server
  ports:
    - port: 80
      targetPort: 8000
      protocol: TCP
  type: ClusterIP


**`deploy_vllm.sh`** · _Block 3 of 7_

!/bin/bash — Deploy vLLM to GKE


In [ ]:
#!/bin/bash
# Deploy vLLM to GKE

# Store HuggingFace token as K8s secret
kubectl create secret generic hf-token \
  --from-literal=token=hf_YOUR_TOKEN_HERE

# Apply the deployment
kubectl apply -f 02_vllm_deployment.yaml

# Watch pod startup (model download + load takes ~3-5 min)
kubectl get pods -l app=vllm-server -w

# Check logs
kubectl logs -l app=vllm-server --tail=50 -f

# Test (port-forward for local testing)
kubectl port-forward svc/vllm-service 8000:80 &

curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "google/gemma-2-9b-it",
    "messages": [{"role": "user", "content": "What is MCP?"}],
    "max_tokens": 200
  }'


> **What just happened?** vLLM deployed as a Kubernetes Deployment serving Gemma 2 9B with: 90% GPU memory utilization (fits the model with room for KV cache), 8192 max context length, OpenAI-compatible API on port 8000. Readiness probe waits 2 minutes for model loading. Termination grace period (120s) finishes in-flight requests before shutdown. HuggingFace token stored as a K8s secret. The Service exposes vLLM on ClusterIP port 80. The API is drop-in compatible with OpenAI's SDK — same /v1/chat/completions endpoint.


### Step 3 · Auto-Scaling — Scale on GPU Utilization + Queue Depth

More requests → more pods → more GPU nodes (automatically).

**`03_autoscaling.yaml`** · _Block 4 of 7_

═══════════════════════════════════════════ — HORIZONTAL POD AUTOSCALER


In [ ]:
# ═══════════════════════════════════════════
# HORIZONTAL POD AUTOSCALER
# Scale vLLM pods based on GPU utilization
# and pending request queue depth.
# ═══════════════════════════════════════════
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: vllm-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: vllm-server
  minReplicas: 1
  maxReplicas: 4
  behavior:
    scaleUp:
      stabilizationWindowSeconds: 60   # wait 1 min before scaling up
      policies:
        - type: Pods
          value: 1                     # add 1 pod at a time
          periodSeconds: 120           # every 2 min max
    scaleDown:
      stabilizationWindowSeconds: 300  # wait 5 min before scaling down
      policies:
        - type: Pods
          value: 1
          periodSeconds: 300
  metrics:
    # Scale when GPU utilization exceeds 70%
    - type: Pods
      pods:
        metric:
          name: duty_cycle   # DCGM GPU utilization metric
        target:
          type: AverageValue
          averageValue: "70"

    # Also scale when average CPU > 60%
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 60


**`scaling_setup.sh`** · _Block 5 of 7_

!/bin/bash — Setup GPU metrics for HPA


In [ ]:
#!/bin/bash
# Setup GPU metrics for HPA

# Install NVIDIA DCGM exporter (GPU metrics → Prometheus format)
kubectl apply -f https://raw.githubusercontent.com/NVIDIA/dcgm-exporter/main/deployment/dcgm-exporter.yaml

# Install custom metrics adapter (bridges Prometheus → K8s HPA)
kubectl apply -f https://raw.githubusercontent.com/GoogleCloudPlatform/k8s-stackdriver/master/custom-metrics-stackdriver-adapter/deploy/production/adapter.yaml

# Apply the HPA
kubectl apply -f 03_autoscaling.yaml

# Watch scaling behavior
kubectl get hpa vllm-hpa -w

echo "
  Scaling behavior:
    1 pod  → handles ~50 concurrent requests
    2 pods → handles ~100 concurrent requests
    4 pods → handles ~200 concurrent requests

    Scale up:  GPU util > 70% for 1 minute → +1 pod (every 2 min)
    Scale down: GPU util < 50% for 5 minutes → -1 pod

    Node autoscaler adds GPU nodes automatically when
    pods are pending due to insufficient GPU resources.
"


> **What just happened?** HPA scales vLLM pods from 1 to 4 based on GPU utilization (target: 70%) and CPU (target: 60%). Scale-up: GPU > 70% for 1 minute → add 1 pod (max every 2 minutes). Scale-down: wait 5 minutes of low utilization before removing a pod (prevents flapping). DCGM exporter provides GPU metrics. The cluster autoscaler works with the HPA: when a new vLLM pod needs a GPU but none is available, GKE automatically provisions a new GPU node. 1 pod ≈ 50 concurrent users. 4 pods ≈ 200 concurrent users.


### Step 4 · Load Balancing + Ingress — HTTPS with Health Checks

Expose vLLM externally with TLS, health-check routing, and request distribution.

**`04_ingress.yaml`** · _Block 6 of 7_

═══════════════════════════════════════════ — INGRESS: HTTPS load balancer


In [ ]:
# ═══════════════════════════════════════════
# INGRESS: HTTPS load balancer
# Routes traffic to healthy vLLM pods.
# ═══════════════════════════════════════════
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: vllm-ingress
  annotations:
    kubernetes.io/ingress.class: "gce"
    kubernetes.io/ingress.global-static-ip-name: "vllm-ip"
    networking.gke.io/managed-certificates: "vllm-cert"
    # Backend health check config
    cloud.google.com/backend-config: '{"default": "vllm-backend-config"}'
spec:
  rules:
    - host: llm.netsetos.com
      http:
        paths:
          - path: /v1/*
            pathType: ImplementationSpecific
            backend:
              service:
                name: vllm-service
                port:
                  number: 80

---
# Backend config: health checks + timeout
apiVersion: cloud.google.com/v1
kind: BackendConfig
metadata:
  name: vllm-backend-config
spec:
  healthCheck:
    type: HTTP
    requestPath: /health
    port: 8000
    checkIntervalSec: 15
    timeoutSec: 5
  timeoutSec: 120   # LLM calls can take up to 2 min
  connectionDraining:
    drainingTimeoutSec: 60

---
# Managed TLS certificate
apiVersion: networking.gke.io/v1
kind: ManagedCertificate
metadata:
  name: vllm-cert
spec:
  domains:
    - llm.netsetos.com


> **What just happened?** Google Cloud Load Balancer in front of vLLM: HTTPS with a managed TLS certificate for llm.netsetos.com . Health checks every 15 seconds on /health — unhealthy pods are removed from rotation. 120-second backend timeout (LLM calls can be slow). Connection draining (60s) finishes in-flight requests during pod shutdown. Traffic is distributed across all healthy vLLM pods automatically.


### Step 5 · Python Client — Use OpenAI SDK to Talk to vLLM

vLLM speaks the OpenAI protocol. Use the openai library — just change the base URL.

**`vllm_client.py`** · _Block 7 of 7_

vLLM CLIENT using OpenAI SDK — Same code that works with OpenAI GPT works


In [ ]:
# =============================================
# vLLM CLIENT using OpenAI SDK
# Same code that works with OpenAI GPT works
# with vLLM — just change the base_url.
# pip install openai
# =============================================

from openai import OpenAI
import time

# ── Connect to vLLM ──
client = OpenAI(
    base_url="https://llm.netsetos.com/v1",  # ← your vLLM endpoint
    api_key="not-needed",                     # ← vLLM doesn't require a key
)

# ── 1. Chat completion ──
def chat(prompt: str, model: str = "google/gemma-2-9b-it") -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.7,
    )
    return response.choices[0].message.content

# ── 2. Streaming ──
def stream_chat(prompt: str):
    stream = client.chat.completions.create(
        model="google/gemma-2-9b-it",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        max_tokens=300,
    )
    for chunk in stream:
        token = chunk.choices[0].delta.content or ""
        print(token, end="", flush=True)
    print()

# ── 3. Batch for throughput ──
def batch_chat(prompts: list[str]) -> list[str]:
    """Send multiple requests. vLLM batches them internally."""
    import asyncio
    from openai import AsyncOpenAI

    async_client = AsyncOpenAI(base_url="https://llm.netsetos.com/v1", api_key="x")

    async def _call(p):
        r = await async_client.chat.completions.create(
            model="google/gemma-2-9b-it",
            messages=[{"role": "user", "content": p}], max_tokens=200)
        return r.choices[0].message.content

    async def _batch():
        return await asyncio.gather(*[_call(p) for p in prompts])

    return asyncio.run(_batch())

# ── 4. Benchmark ──
def benchmark():
    prompts = [f"Explain concept {i} in 1 sentence." for i in range(10)]

    # Sequential
    start = time.time()
    [chat(p) for p in prompts]
    seq_time = time.time() - start

    # Batch (concurrent)
    start = time.time()
    batch_chat(prompts)
    batch_time = time.time() - start

    print(f"\n  Benchmark (10 queries):")
    print(f"    Sequential: {seq_time:.1f}s")
    print(f"    Batch:      {batch_time:.1f}s")
    print(f"    Speedup:    {seq_time/batch_time:.1f}x")
    print(f"    Throughput:  {10/batch_time:.1f} req/s")

# ── Demo ──
print("1. Chat:")
print(f"   {chat('What is vLLM? One sentence.')}\n")

print("2. Streaming:")
stream_chat("Count from 1 to 5.")

print("\n3. Benchmark:")
benchmark()


> **What just happened?** The openai Python SDK talks to vLLM by changing one line : base_url="https://llm.netsetos.com/v1" . Same chat.completions.create() , same streaming, same response format. Async batch sends 10 requests concurrently — vLLM's continuous batching processes them together on the GPU, achieving 3-5x higher throughput than sequential. Any code that works with OpenAI's API works with vLLM — zero changes to your application logic.


---

## Wrap-up

You've walked through all 7 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
